In [ ]:
# Step 1: Install PySpark
!pip install pyspark

In [ ]:
# Step 2: Import Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
# Step 3: Create Spark Session
spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Ans:
Traditional MapReduce suffers from:
Disk-based processing causing high latency.
Repeated read/write operations.
Inefficient iterative computations.
Complex programming model.

Spark is preferred because:
It uses in-memory computation.
Provides faster execution.
Supports SQL, Machine Learning, and Streaming.
Offers easy-to-use DataFrame APIs.

Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.
Ans: Spark stores intermediate data in memory instead of writing it to disk after each step. This significantly reduces disk I/O and speeds up iterative machine learning algorithms.

Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [54]:
df = df.dropDuplicates(["user_id", "transaction_date"])

Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [42]:
from pyspark.sql.functions import avg, col

df.filter(
    col("region") == "West"
).groupBy(
    "product_category"
).agg(
    avg("sale_amount").alias("Average_Sale")
).show()

+----------------+-----------------+
|product_category|     Average_Sale|
+----------------+-----------------+
|     Electronics|533.3333333333334|
|        Clothing|            250.0|
+----------------+-----------------+



Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

ANS- Theory
.na.drop() removes rows containing null values.
.na.fill() replaces null values with specified values.

In [44]:
df = df.na.fill(
    "Unknown",
    subset=["status"]
)

In [30]:
df = df.na.fill(
    0,
    subset=["price"]
)

Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [33]:
# Step 10: Count Records by City
df.groupBy(
    "city"
).count().show()

+-----------+-----+
|       city|count|
+-----------+-----+
|    Phoenix|    1|
|Los Angeles|    2|
|    Chicago|    1|
|    Houston|    2|
|   New York|    1|
+-----------+-----+



Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

Ans- Spark DataFrames are immutable. Operations such as dropping columns or renaming columns create a new DataFrame instead of modifying the original DataFrame.

In [46]:
new_df = df.drop("age")

new_df = df.withColumnRenamed(
    "price",
    "sale_price"
)

Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [47]:
df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
).show()

+-------+----------------+-----+---+------------+------+----------------+-----------+---------+-----------+---------------+--------+-----+--------+-------------------+
|user_id|transaction_date| name|age|subscription|region|product_category|sale_amount|   status|       city|          email|username|price|store_id|         event_time|
+-------+----------------+-----+---+------------+------+----------------+-----------+---------+-----------+---------------+--------+-----+--------+-------------------+
|    101|      2025-08-01| John| 25|     Premium|  West|     Electronics|        500|Completed|   New York| john@gmail.com| john123|  500|       1|2025-08-01 10:30:00|
|    103|      2025-08-03|  Bob| 22|     Premium|  West|     Electronics|        700|Completed|Los Angeles|  bob@gmail.com|  bob789|  700|       1|2025-08-03 09:45:00|
|    104|      2025-08-04|David| 28|     Premium| South|       Furniture|        450|  Pending|    Houston|david@gmail.com|david123|  450|       3|2025-08-04 14

Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?
Ans- Handling null values before applying sum() or avg() prevents inaccurate calculations and ensures reliable analytical results.

Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

Ans- The raw_timestamp column was converted to TimestampType and renamed to event_time

In [49]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- status: string (nullable = false)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [50]:
df.select("event_time").show(truncate=False)

+-------------------+
|event_time         |
+-------------------+
|2025-08-01 10:30:00|
|2025-08-02 11:15:00|
|2025-08-03 09:45:00|
|2025-08-04 14:20:00|
|2025-08-05 08:30:00|
|2025-08-08 15:00:00|
|2025-08-09 16:20:00|
+-------------------+



Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

Ans- Shuffle is the process of redistributing data across partitions during operations like groupBy(). It is considered a wide transformation because data moves between partitions, increasing execution cost.

Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [51]:
df = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [52]:
from pyspark.sql.functions import min,max,mean

df.agg(
    min("price").alias("Minimum"),
    max("price").alias("Maximum"),
    mean("price").alias("Average")
).show()

+-------+-------+-----------------+
|Minimum|Maximum|          Average|
+-------+-------+-----------------+
|      0|    700|385.7142857142857|
+-------+-------+-----------------+



Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

Ans- Using inferSchema=True on messy or inconsistent date formats may result in incorrect data types or conversion errors. Defining the schema manually is safer.

In [56]:
df = spark.read.csv(
    "dataset.csv",
    header=True,
    inferSchema=True
)

df.show()

+-------+----------------+-----+---+------------+------+----------------+-----------+---------+-----------+---------------+--------+-----+--------+-------------------+
|user_id|transaction_date| name|age|subscription|region|product_category|sale_amount|   status|       city|          email|username|price|store_id|      raw_timestamp|
+-------+----------------+-----+---+------------+------+----------------+-----------+---------+-----------+---------------+--------+-----+--------+-------------------+
|    101|      2025-08-01| John| 25|     Premium|  West|     Electronics|        500|Completed|   New York| john@gmail.com| john123|  500|       1|2025-08-01 10:30:00|
|    102|      2025-08-02|Alice| 30|       Basic|  East|        Clothing|        300|     NULL|    Chicago|alice@gmail.com|alice456|  300|       2|2025-08-02 11:15:00|
|    103|      2025-08-03|  Bob| 22|     Premium|  West|     Electronics|        700|Completed|Los Angeles|  bob@gmail.com|  bob789|  700|       1|2025-08-03 09

Q15: Write a final processing pipeline that:

Filters out duplicates.

Fills null prices with 0.

Groups by store_id to calculate total revenue.

In [53]:
from pyspark.sql.functions import sum

# Remove duplicates
df1 = df.dropDuplicates()

# Fill null prices with zero
df2 = df1.na.fill(
    0,
    subset=["price"]
)

# Calculate total revenue
result = df2.groupBy(
    "store_id"
).agg(
    sum("price").alias("total_revenue")
)

result.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       1|         1200|
|       3|          450|
|       2|         1050|
+--------+-------------+



In [57]:
import os

os.makedirs("output", exist_ok=True)

In [58]:
result.toPandas().to_csv(
    "output/results.csv",
    index=False
)

In [59]:
!ls output

results.csv


In [60]:
from google.colab import files

files.download("output/results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>